# Movie Metadata Cleaning

Profile `movies_metadata.csv` and clean the core ID, date, financial, runtime, and popularity columns. The source file is left unchanged.


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'data' / 'raw' / 'movies_metadata.csv').exists()),
    Path.cwd(),
)
INPUT_PATH = PROJECT_ROOT / 'data' / 'raw' / 'movies_metadata.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'movies_metadata.csv'
TARGET_COLUMNS = ['id', 'release_date', 'budget', 'revenue', 'runtime', 'popularity']

raw = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Input: {INPUT_PATH}')
print(f'Rows: {len(raw):,}; columns: {len(raw.columns):,}')

Input: /Users/jasper/css-dsml-fall2027/data/raw/movies_metadata.csv
Rows: 45,466; columns: 24


## Raw data profile


In [2]:
print('Shape:', raw.shape)
print('\nData types:')
print(raw.dtypes.to_string())
print('\nMissing values by column:')
print(raw.isna().sum().sort_values(ascending=False).to_string())
duplicate_extras = int(raw.duplicated().sum())
duplicate_group_rows = int(raw.duplicated(keep=False).sum())
print(f'\nExact duplicate rows beyond first copies: {duplicate_extras:,}')
print(f'Rows belonging to exact-duplicate groups: {duplicate_group_rows:,}')
duplicate_rows = raw.loc[raw.duplicated(keep=False)]
if not duplicate_rows.empty:
    display(duplicate_rows.head(20))
print('\nTarget-column missing values:')
print(raw[TARGET_COLUMNS].isna().sum().to_string())

Shape: (45466, 24)

Data types:
adult                     object
belongs_to_collection     object
budget                    object
genres                    object
homepage                  object
id                        object
imdb_id                   object
original_language         object
original_title            object
overview                  object
popularity                object
poster_path               object
production_companies      object
production_countries      object
release_date              object
revenue                  float64
runtime                  float64
spoken_languages          object
status                    object
tagline                   object
title                     object
video                     object
vote_average             float64
vote_count               float64

Missing values by column:
belongs_to_collection    40972
homepage                 37684
tagline                  25054
overview                   954
poster_path              

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
676,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,105045,tt0111613,de,Das Versprechen,"East-Berlin, 1961, shortly after the erection ...",...,1995-02-16,0.0,115.0,"[{'iso_639_1': 'de', 'name': 'Deutsch'}]",Released,"A love, a hope, a wall.",The Promise,False,5.0,1.0
1465,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,105045,tt0111613,de,Das Versprechen,"East-Berlin, 1961, shortly after the erection ...",...,1995-02-16,0.0,115.0,"[{'iso_639_1': 'de', 'name': 'Deutsch'}]",Released,"A love, a hope, a wall.",The Promise,False,5.0,1.0
7345,False,NaN,0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",NaN,5511,tt0062229,fr,Le Samouraï,Hitman Jef Costello is a perfectionist who alw...,...,1967-10-25,39481.0,105.0,"[{'iso_639_1': 'fr', 'name': 'Français'}]",Released,There is no solitude greater than that of the ...,Le Samouraï,False,7.9,187.0
8068,False,NaN,0,"[{'id': 12, 'name': 'Adventure'}, {'id': 16, '...",NaN,23305,tt0295682,en,The Warrior,"In feudal India, a warrior (Khan) who renounce...",...,2001-09-23,0.0,86.0,"[{'iso_639_1': 'hi', 'name': 'हिन्दी'}]",Released,NaN,The Warrior,False,6.3,15.0
9165,False,NaN,0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",NaN,5511,tt0062229,fr,Le Samouraï,Hitman Jef Costello is a perfectionist who alw...,...,1967-10-25,39481.0,105.0,"[{'iso_639_1': 'fr', 'name': 'Français'}]",Released,There is no solitude greater than that of the ...,Le Samouraï,False,7.9,187.0
9327,False,NaN,0,"[{'id': 12, 'name': 'Adventure'}, {'id': 16, '...",NaN,23305,tt0295682,en,The Warrior,"In feudal India, a warrior (Khan) who renounce...",...,2001-09-23,0.0,86.0,"[{'iso_639_1': 'hi', 'name': 'हिन्दी'}]",Released,NaN,The Warrior,False,6.3,15.0
13220,False,"{'id': 158365, 'name': 'Why We Fight', 'poster...",0,"[{'id': 99, 'name': 'Documentary'}]",NaN,159849,tt0173769,en,Why We Fight: Divide and Conquer,"The third film of Frank Capra's 'Why We Fight""...",...,1943-01-01,0.0,57.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Why We Fight: Divide and Conquer,False,5.0,1.0
13261,False,NaN,0,"[{'id': 53, 'name': 'Thriller'}, {'id': 9648, ...",NaN,141971,tt1180333,fi,Blackout,Recovering from a nail gun shot to the head an...,...,2008-12-26,0.0,108.0,"[{'iso_639_1': 'fi', 'name': 'suomi'}]",Released,Which one is the first to return - memory or t...,Blackout,False,6.7,3.0
13375,False,NaN,0,"[{'id': 53, 'name': 'Thriller'}, {'id': 9648, ...",NaN,141971,tt1180333,fi,Blackout,Recovering from a nail gun shot to the head an...,...,2008-12-26,0.0,108.0,"[{'iso_639_1': 'fi', 'name': 'suomi'}]",Released,Which one is the first to return - memory or t...,Blackout,False,6.7,3.0
13603,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 35, 'name...",NaN,168538,tt0084387,en,Nana,"In Zola's Paris, an ingenue arrives at a tony ...",...,1983-06-13,0.0,92.0,[],Released,NaN,"Nana, the True Key of Pleasure",False,4.7,3.0



Target-column missing values:
id                0
release_date     87
budget            0
revenue           6
runtime         263
popularity        5


## Target-column checks

Invalid values are coerced to missing. Zero budget, revenue, and runtime values are treated as unavailable-value sentinels; zero popularity is retained.


In [3]:
for column in ['id', 'budget', 'revenue', 'runtime', 'popularity']:
    parsed = pd.to_numeric(raw[column], errors='coerce')
    invalid_count = int((raw[column].notna() & parsed.isna()).sum())
    zero_count = int(parsed.eq(0).sum())
    print(f'{column}: invalid nonmissing={invalid_count:,}; zero={zero_count:,}')

parsed_dates = pd.to_datetime(raw['release_date'], errors='coerce')
invalid_dates = raw['release_date'].notna() & parsed_dates.isna()
print(f'release_date: invalid nonmissing={int(invalid_dates.sum()):,}')
print('Invalid ID examples:')
numeric_ids = pd.to_numeric(raw['id'], errors='coerce')
display(raw.loc[raw['id'].notna() & numeric_ids.isna(), ['id', 'original_title', 'release_date']])
print('Invalid date examples:')
display(raw.loc[invalid_dates, ['id', 'original_title', 'release_date']])

id: invalid nonmissing=3; zero=0
budget: invalid nonmissing=3; zero=36,573
revenue: invalid nonmissing=0; zero=38,052
runtime: invalid nonmissing=0; zero=1,558
popularity: invalid nonmissing=1; zero=66
release_date: invalid nonmissing=3
Invalid ID examples:


,id,original_title,release_date
19730,1997-08-20,"[{'iso_639_1': 'en', 'name': 'English'}]",1
29503,2012-09-29,"[{'iso_639_1': 'ja', 'name': '日本語'}]",12
35587,2014-01-01,"[{'iso_639_1': 'en', 'name': 'English'}]",22


Invalid date examples:


,id,original_title,release_date
19730,1997-08-20,"[{'iso_639_1': 'en', 'name': 'English'}]",1
29503,2012-09-29,"[{'iso_639_1': 'ja', 'name': '日本語'}]",12
35587,2014-01-01,"[{'iso_639_1': 'en', 'name': 'English'}]",22


## Cleaning decisions

The file contains three structurally shifted records: their IDs are dates and their date fields are not parseable dates. These values are set to missing rather than inferred or repaired. Rows and exact duplicates are retained; only the six requested columns are converted.


In [4]:
clean = raw.copy()
clean['id'] = pd.to_numeric(clean['id'], errors='coerce').astype('Int64')
clean['release_date'] = pd.to_datetime(clean['release_date'], errors='coerce')

for column in ['budget', 'revenue', 'runtime']:
    values = pd.to_numeric(clean[column], errors='coerce')

clean['popularity'] = pd.to_numeric(clean['popularity'], errors='coerce').astype('float64')
assert len(clean) == len(raw)
assert clean.columns.equals(raw.columns)
assert clean['id'].isna().sum() == 3
assert clean['release_date'].isna().sum() == 90
print('Cleaned target dtypes:')
print(clean[TARGET_COLUMNS].dtypes.to_string())
print('\nCleaned target missing values:')
print(clean[TARGET_COLUMNS].isna().sum().to_string())
print(f'\nRows preserved: {len(clean):,}; exact duplicate rows retained: {int(clean.duplicated().sum()):,}')

Cleaned target dtypes:
id                       Int64
release_date    datetime64[ns]
budget                  object
revenue                float64
runtime                float64
popularity             float64

Cleaned target missing values:
id                3
release_date     90
budget            0
revenue           6
runtime         263
popularity        6

Rows preserved: 45,466; exact duplicate rows retained: 17


## Save and verify the processed copy


In [5]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
clean[TARGET_COLUMNS].to_csv(OUTPUT_PATH, index=False, date_format='%Y-%m-%d')
verified = pd.read_csv(OUTPUT_PATH, low_memory=False)
print(f'Saved: {OUTPUT_PATH}')
print('Verified shape:', verified.shape)
print('Verified target dtypes after CSV reload:')
print(verified[TARGET_COLUMNS].dtypes.to_string())
assert verified['id'].isna().sum() == clean['id'].isna().sum()
assert verified['release_date'].isna().sum() == clean['release_date'].isna().sum()
assert verified['budget'].isna().sum() == clean['budget'].isna().sum()
assert verified['revenue'].isna().sum() == clean['revenue'].isna().sum()
assert verified['runtime'].isna().sum() == clean['runtime'].isna().sum()
assert verified['popularity'].isna().sum() == clean['popularity'].isna().sum()
print('Output checks passed.')

Saved: /Users/jasper/css-dsml-fall2027/data/processed/movies_metadata.csv
Verified shape: (45466, 6)
Verified target dtypes after CSV reload:
id              float64
release_date     object
budget           object
revenue         float64
runtime         float64
popularity      float64
Output checks passed.
